## MMR TIME

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma,FAISS


#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os
import numpy as np
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

C:\Users\kanha\AppData\Local\Temp\ipykernel_6164\3387885773.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
loader=TextLoader("langchain_rag_dataset.txt")
raw_docs=loader.load()
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [4]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
vectorstore_diff=FAISS.from_documents(chunks,embeddings)
retriever_diff=vectorstore_diff.as_retriever(search_type="mmr",search_kwargs={"k":8})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
template="""
Ansert the question based on following context:
{context}
Question:{input}
"""
prompt=PromptTemplate.from_template(template)
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key="")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024BA4921B50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024BA4A1D410>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
### Create stiff document chain
document_chain=create_stuff_documents_chain(llm=llm,prompt=prompt)

rag_chain=create_retrieval_chain(retriever_diff,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000024BDA95C8D0>, search_type='mmr', search_kwargs={'k': 8}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnsert the question based on following context:\n{context}\nQuestion:{input}\n')
            | ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Ver

In [8]:
query={"input":"How can i build an app using LLMs?"}
response=rag_chain.invoke(query)
print(f"Answer:\n {response['answer']}")

for i,doc in enumerate(response['context']):
    print(f"\n Doc {i+1} :{doc.page_content}")

Answer:
 To build an app using Large Language Models (LLMs), you can utilize the LangChain framework, which simplifies the development of LLM-based applications. Here's a step-by-step guide:

1. **Use LangChain's abstractions**: LangChain provides abstractions for working with prompts, chains, memory, and agents. These abstractions will help you build complex LLM-based systems more easily.
2. **Choose a chain**: Select a suitable chain for your application, such as:
	* 'stuff' chain for short documents
	* 'map-reduce' chain for large documents
	* 'refine' chain for iterative updates
3. **Utilize reranking and retrieval**: Improve context quality by using LLMs or neural cross-encoders for reranking retrieved results. You can also use Retrieval-Augmented Generation (RAG) pipelines for document loading, splitting, embedding, retrieval, and response generation.
4. **Leverage agents and memory**: Use LangChain agents to decide which tools to call and in what order during a task. You can als